## Step 1: Install Required Libraries

Install the Semantic Link Labs library, which provides functions for creating and managing Direct Lake semantic models in Microsoft Fabric.

In [ ]:
#%pip install -q semantic-link-labs

import importlib, subprocess, sys

if importlib.util.find_spec("sempy_labs") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "semantic-link-labs"])

## Step 2: Import Libraries and Set Variables

Import the required Python libraries and define variables for the lakehouse name and semantic model name used throughout this lab.

In [ ]:
import sempy_labs as labs
from sempy import fabric
import sempy
import pandas
import json
import time

#LakehouseName = "AdventureWorks"
#SemanticModelName = f"{LakehouseName}_model"

In [ ]:
%run 2. Parameters


## Step 3: Create or Connect to Lakehouse

Check whether the AdventureWorks lakehouse already exists. If not, create it. Then retrieve the workspace and lakehouse identifiers needed for subsequent steps.

In [ ]:
# Only check lakehouses in current workspace
current_workspace_id = fabric.resolve_workspace_id()
lakehouses = labs.list_lakehouses(workspace=current_workspace_id)

print(f"Current workspace: {current_workspace_id}")
print(f"Looking for lakehouse: '{lakehouse_name}'")
print(lakehouses[["Lakehouse Name"]])

client = fabric.FabricRestClient()

if lakehouse_name in lakehouses["Lakehouse Name"].values:
    print(f"Lakehouse '{lakehouse_name}' found, getting ID...")
else:
    print(f"Lakehouse '{lakehouse_name}' not found, creating it...")
    create_resp = client.post(
        f"/v1/workspaces/{current_workspace_id}/lakehouses",
        json={"displayName": lakehouse_name}
    )
    print(f"Create response: {create_resp.status_code} - {create_resp.text}")
    time.sleep(20)

# Get lakehouse ID via REST API instead of notebookutils
items = client.get(f"/v1/workspaces/{current_workspace_id}/lakehouses").json()
lakehouse = next((l for l in items["value"] if l["displayName"] == lakehouse_name), None)

if lakehouse:
    lakehouseId = lakehouse["id"]
    workspaceId = current_workspace_id
    workspaceName = sempy.fabric.resolve_workspace_name(workspaceId)
    print(f"WorkspaceId = {workspaceId}, LakehouseID = {lakehouseId}, Workspace Name = {workspaceName}")
else:
    raise Exception(f"Lakehouse '{lakehouse_name}' could not be found or created")

In [ ]:
lakehouses = labs.list_lakehouses()

if lakehouse_name in lakehouses["Lakehouse Name"].values:
    lakehouseId = notebookutils.lakehouse.getWithProperties(lakehouse_name)["id"]
else:
    lakehouseId = fabric.create_lakehouse(lakehouse_name)

workspaceId = notebookutils.lakehouse.getWithProperties(lakehouse_name)["workspaceId"]
workspaceName = sempy.fabric.resolve_workspace_name(workspaceId)
print(f"WorkspaceId = {workspaceId}, LakehouseID = {lakehouseId}, Workspace Name = {workspaceName}")